# Experiment 1: Main Benchmark (Google Colab)

**What this does:** Compares 3 algorithms on a synthetic low-rank bandit problem:
- **SPSC Algorithm 1** (ours) — learns the subspace via probes, then exploits in low dimension
- **LinUCB** — standard bandit baseline, works in full ambient dimension
- **Oracle-LinUCB** — cheats by knowing the true subspace (unattainable ceiling)

**Expected result:** SPSC should beat LinUCB and approach Oracle performance.

**Runtime:** ~5 minutes on Colab.

## Step 1: Upload Code Files

Run the cell below. A **"Choose Files"** button will appear.

Click it and select **both** of these files from your `C:\lowrank\Code` folder:
1. `algorithm.py`
2. `environment.py`

You can select both at once by holding **Ctrl** while clicking.

In [ ]:
from google.colab import files

print("Click 'Choose Files' and select BOTH:")
print("  1. algorithm.py")
print("  2. environment.py")
print("(Hold Ctrl to select both at once)\n")

uploaded = files.upload()

# Verify
import os
for f in ['algorithm.py', 'environment.py']:
    if os.path.exists(f):
        print(f"  {f} uploaded OK")
    else:
        print(f"  WARNING: {f} is missing! Upload it before continuing.")

## Step 2: Check Dependencies
Colab already has everything we need. This just confirms it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys

print(f"Python:     {sys.version.split()[0]}")
print(f"NumPy:      {np.__version__}")
print(f"Matplotlib: {plt.matplotlib.__version__}")
print("\nAll dependencies OK!")

## Step 3: Import Our Code
This imports the environment (synthetic bandit problem) and the 3 algorithms.

In [ ]:
from environment import LowRankLDSEnvironment
from algorithm import SPSC_Algorithm1, LinUCB, OracleLinUCB

print("Imported successfully!")

## Step 4: Set Parameters
These match the paper exactly:
- `D=4`: ambient dimension (the full space the bandit operates in)
- `R=1`: true latent rank (the hidden low-dimensional structure)
- `K=4`: number of regime changes (the environment shifts 4 times)
- `T=6000`: total rounds
- `N_SEEDS=10`: how many random repetitions (for statistical confidence)

In [ ]:
D           = 4      # ambient dimension
R           = 1      # latent rank
K           = 4      # number of segments
T           = 6000   # horizon (total rounds)
PROBE_EVERY = 30     # SPSC probes every 30 rounds
PROBE_COST  = 0.1    # cost per probe round
WINDOW      = 100    # sliding window for regression
N_SEEDS     = 10     # number of random seeds

print(f"Setup: d={D}, r={R}, K={K}, T={T}")
print(f"SPSC: probe_every={PROBE_EVERY}, cost={PROBE_COST}, window={WINDOW}")
print(f"Seeds: {N_SEEDS}")

## Step 5: Quick Sanity Check (1 seed)
Before running all 10 seeds, let's do a quick single run to make sure everything works.
This should take about 30 seconds.

In [ ]:
# Create the environment with seed=0
env_test = LowRankLDSEnvironment(d=D, r=R, K=K, T=T, seed=0)

print(f"Environment created:")
print(f"  Dimensions: d={env_test.d}, r={env_test.r}")
print(f"  Segments: K={env_test.K}, boundaries at {env_test.tau}")
print(f"  Horizon: T={env_test.T}")
print(f"  LDS spectral radius: {env_test.spectral_radius}")
print(f"  Noise sigma: {env_test.sigma_eps}")

In [ ]:
# Run SPSC on this single environment
env = LowRankLDSEnvironment(d=D, r=R, K=K, T=T, seed=0)
spsc_test = SPSC_Algorithm1(
    env, probe_every=PROBE_EVERY, probe_cost=PROBE_COST,
    window=WINDOW, lam=1.0, delta=0.05, seed=0
).run()

print(f"SPSC single run done!")
print(f"  Final costed regret:  {spsc_test.cumulative_costed_regret[-1]:.1f}")
print(f"  Final control regret: {spsc_test.cumulative_control_regret[-1]:.1f}")
print(f"  Total probes used:    {spsc_test.total_probes}")
print("\nSanity check passed! Ready to run full experiment.")

## Step 6: Run Full Experiment (All 10 Seeds)
This runs all 3 algorithms across 10 random seeds. **Takes ~5 minutes on Colab.**

You'll see progress printed as each seed completes.

In [ ]:
def make_env(seed):
    return LowRankLDSEnvironment(d=D, r=R, K=K, T=T, seed=seed * 100)

# Storage for results
results = {
    "SPSC-Alg1": [],
    "LinUCB": [],
    "Oracle-LinUCB": [],
}

for seed in range(N_SEEDS):
    print(f"  Running seed {seed+1}/{N_SEEDS} ...", flush=True)

    # SPSC Algorithm 1
    env = make_env(seed)
    results["SPSC-Alg1"].append(
        SPSC_Algorithm1(env, probe_every=PROBE_EVERY, probe_cost=PROBE_COST,
                        window=WINDOW, lam=1.0, delta=0.05, seed=seed).run()
    )

    # LinUCB (full-dimensional baseline)
    env = make_env(seed)
    results["LinUCB"].append(
        LinUCB(env, lam=1.0, delta=0.05, seed=seed + 1000).run()
    )

    # Oracle-LinUCB (knows true subspace — ceiling)
    env = make_env(seed)
    results["Oracle-LinUCB"].append(
        OracleLinUCB(env, window=WINDOW, lam=1.0, delta=0.05,
                     seed=seed + 2000).run()
    )

print("\nAll seeds complete!")

## Step 7: Results Table
Shows final cumulative regret for each algorithm (mean +/- standard error).

In [ ]:
env_ref = make_env(0)

print("=" * 72)
print("Experiment 1 — Final cumulative regret summary")
print(f"d={D}, r={R}, K={K}, T={T}, probe_every={PROBE_EVERY}, W={WINDOW}, c={PROBE_COST}")
print("-" * 72)
print(f"{'Algorithm':<22}  {'Costed (mean +/- SE)':>20}  {'Control (mean +/- SE)':>20}  {'Probes':>7}")
print("-" * 72)

finals = {}
for name, runs in results.items():
    costed  = np.array([r.cumulative_costed_regret[-1]  for r in runs])
    control = np.array([r.cumulative_control_regret[-1] for r in runs])
    probes  = np.array([r.total_probes                  for r in runs])
    finals[name] = costed.mean()
    print(
        f"  {name:<20}  "
        f"{costed.mean():>8.1f} +/- {costed.std()/np.sqrt(N_SEEDS):>5.1f}  "
        f"{control.mean():>8.1f} +/- {control.std()/np.sqrt(N_SEEDS):>5.1f}  "
        f"{probes.mean():>7.0f}"
    )

print("-" * 72)
spsc = finals["SPSC-Alg1"]
lin  = finals["LinUCB"]
ora  = finals["Oracle-LinUCB"]
print(f"  SPSC / LinUCB ratio  : {spsc/lin:.3f}  ({'SPSC wins' if spsc < lin else 'LinUCB wins'})")
print(f"  SPSC / Oracle ratio  : {spsc/ora:.3f}  (1.0 = oracle-level)")
print(f"  Theory sqrt(r/d)     : {(R/D)**0.5:.3f}  (exploitation-only lower bound)")
print("=" * 72)

## Step 8: Plot the Figure
Two panels:
- **(a) Costed Regret**: includes the probe cost — this is the primary metric
- **(b) Control Regret**: probe cost excluded — shows pure exploitation quality

Shaded bands = +/-1 standard error across seeds. Vertical dotted lines = segment boundaries.

In [ ]:
# Helper to compute mean and standard error across seeds
def agg(runs, attr):
    data = np.stack([getattr(r, attr) for r in runs])
    mean = data.mean(axis=0)
    se   = data.std(axis=0) / np.sqrt(len(runs))
    return mean, se

# Colors and styles
COLORS = {"SPSC-Alg1": "#1f77b4", "LinUCB": "#d62728", "Oracle-LinUCB": "#2ca02c"}
STYLES = {"SPSC-Alg1": "-", "LinUCB": "--", "Oracle-LinUCB": ":"}
LABELS = {
    "SPSC-Alg1":     f"SPSC Algorithm 1 (r={R}-dim)",
    "LinUCB":        f"Ambient LinUCB (d={D}-dim)",
    "Oracle-LinUCB": f"Oracle LinUCB (true subspace, r={R}-dim)",
}

env_ref = make_env(0)
t_axis = np.arange(1, T + 1)
change_pts = env_ref.tau[1:]  # segment boundaries

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.subplots_adjust(wspace=0.30)

panel_info = [
    ("cumulative_costed_regret",  "Cumulative Costed Regret",
     "(a) Cumulative Costed Regret (primary metric)"),
    ("cumulative_control_regret", "Cumulative Control Regret",
     "(b) Cumulative Control Regret (probe cost excluded)"),
]

for ax, (attr, ylabel, title) in zip(axes, panel_info):
    for name, runs in results.items():
        mean, se = agg(runs, attr)
        ax.plot(t_axis, mean, color=COLORS[name], ls=STYLES[name],
                lw=2.0, label=LABELS[name], zorder=3)
        ax.fill_between(t_axis, mean - se, mean + se,
                        color=COLORS[name], alpha=0.18, zorder=2)
    for cp in change_pts:
        ax.axvline(cp, color="gray", ls=":", lw=1.0, alpha=0.65)
    ax.set_xlabel("Round t", fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9, loc="upper left")
    ax.set_xlim(1, T)
    ax.set_ylim(bottom=0)

fig.suptitle(
    f"Experiment 1: SPSC vs LinUCB vs Oracle  |  "
    f"d={D}, r={R}, K={K}, T={T}, c={PROBE_COST}, "
    f"probe_every={PROBE_EVERY}, W={WINDOW}  "
    f"({N_SEEDS} seeds, bands = +/- 1 SE)",
    fontsize=10, y=1.02,
)

plt.tight_layout()
plt.show()

## What to Look For

- **Blue solid line (SPSC)** should be **below** the red dashed line (LinUCB) = SPSC wins
- **Green dotted line (Oracle)** is the floor — nobody can beat it (it knows the true subspace)
- **SPSC / LinUCB ratio < 1** in the table confirms SPSC outperforms LinUCB
- At each **vertical dotted line** (segment boundary), you can see how quickly each method adapts to the new regime

## (Optional) Save the figure to download it

In [ ]:
fig.savefig("experiment1_main_benchmark.png", bbox_inches="tight", dpi=150)
files.download("experiment1_main_benchmark.png")
print("Download started!")